# 05 — Stock lookup

**What this does:** lets you type one ticker and see its current individual-signal, five-block, and composite rankings, plus ranking history charts.

**Run first:** notebook 01 must have produced `data/panel_russell3000.parquet`. Run notebook 02 first if the lookup should include the newest live month.

**Produces:** tables and charts in this notebook. It does not write or change any file.

**Time:** the first setup usually takes about one minute because all five blocks are calculated once. Each ticker after that should take seconds.

## Step 1 — Load and score the panel once

Run this cell when opening the notebook. The scoring context is reused when you change tickers, so do not rerun it for every company.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'config.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project folder containing config.py.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model.stock_explainer import get_ranking_context, stock_rank_history
from src.notebook_support import display_stock_report

stock_context = get_ranking_context()
latest_date = stock_context.panel['date'].max()
print(
    f'Scores ready through {latest_date:%Y-%m-%d}: '    f'{stock_context.panel["permno"].nunique():,} securities available for lookup.'
)

## Step 2 — Type a ticker and run the lookup

Type the ticker between the quotation marks below, then run the cell. Leave the date blank to use the latest panel month. Change the history length if you want more or fewer months. There is no fixed example ticker: the notebook only looks up the name you enter.

In [ ]:
TICKER = ""  # Type a ticker between the quotes, for example your current research name.
LOOKUP_DATE = None  # None means the latest month.
HISTORY_MONTHS = 60

if not TICKER.strip():
    print('No lookup run: type a ticker in TICKER, then run this cell again.')
else:
    stock_report = stock_rank_history(
        TICKER.strip().upper(),
        date=LOOKUP_DATE,
        months=HISTORY_MONTHS,
        context=stock_context,
    )
    display_stock_report(stock_report)

## How to read the output

Percentile 100 is best and decile 1 is best. The current tables show exactly which raw signals and blocks help or hurt the stock. The charts show whether the current ranking is persistent or a recent change. A missing value is information: it can reflect an economic exclusion, insufficient history, or unavailable accounting data. It is not silently treated as average.

Use the result to frame questions for a fundamental analyst. A high score is not an instruction to buy, and a low score is not a substitute for understanding the company.

## What was produced and what comes next

The notebook displayed the latest rankings and history charts for the ticker you entered. It wrote no files. Change `TICKER` and rerun only Step 2 to look up another company.

For proposed portfolio weights and trades, use **02_monthly_update.ipynb**. For model-level performance, use **04_backtest.ipynb**.

## If something fails

- **Ticker not found:** confirm the symbol and lookup date. The company may not exist in the CRSP-linked panel or may use a historical ticker. Try its PERMNO only if you know it.
- **Missing panel:** run notebook 01.
- **The newest month is absent:** run notebook 02, then restart this notebook so the scoring context reloads.
- **Undefined `stock_context`:** run Step 1 before Step 2.
- **Charts appear twice:** restart the kernel and run from Step 1; the display helper is designed to render each chart once. See `START_HERE.ipynb` for detailed troubleshooting.